# GLM and Permutation Tests

In [9]:
import sys
import os
sys.path.append(os.path.abspath(".."))
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
from statsmodels.stats.multitest import multipletests
from analyses.glm_permutation_tests import run_glm, plot_glm_coefficients, run_permutation_anova, \
    plot_permutation_anova_results_summary, merge_glm_and_permutation_anova
# Notebook header
import pandas as pd
from analyses.spike_count import prepare_binned_spike_data, prepare_exploded_spike_data, aggregate_trial_level
import warnings
from scipy.stats import ConstantInputWarning
warnings.simplefilter("ignore", ConstantInputWarning)

## Load Session for Analysis

In [4]:
# Step 1: Load and prepare data
date = "2023-09-26"
round_no = 3
bin_size = 0.05

analysis_df = prepare_binned_spike_data(date, round_no, bin_size)


Reading Intan Technologies RHD2000 Data File, Version 3.2

Found 24 amplifier channels.
Found 0 auxiliary input channels.
Found 0 supply voltage channels.
Found 0 board ADC channels.
Found 2 board digital input channels.
Found 0 board digital output channels.
Found 0 temperature sensors channels.

Header file contains no data.  Amplifiers were sampled at 20.00 kS/s.
Done!  Elapsed time: 0.0 seconds


## Effect of Stimulus Identity for Zombies

In [38]:
zombies_df = analysis_df[analysis_df['MonkeyGroup'] == 'Zombies']
formula = "SpikeCount ~ C(MonkeyName)"  # Stimulus identity


## GLM

In [34]:
glm_results = run_glm(zombies_df, formula=formula)
print(glm_results.head())

Running GLM per neuron: 100%|██████████| 32/32 [00:00<00:00, 43.29it/s]

                   index      Coef.      Std.Err.             z     P>|z|  \
0              Intercept -28.128402  38133.918272 -7.376216e-04  0.999411   
1  C(MonkeyName)[T.143H]  23.364094  38133.918275  6.126854e-04  0.999511   
2  C(MonkeyName)[T.151J]  -0.000007  55501.364509 -1.290195e-10  1.000000   
3   C(MonkeyName)[T.67G]  -0.000007  53962.051818 -1.327011e-10  1.000000   
4   C(MonkeyName)[T.69X]  23.927983  38133.918274  6.274724e-04  0.999499   

          [0.025         0.975]                           NeuronID  
0  -74769.234805   74712.978001  2023-09-26_3_Channel.C_012_Unit 1  
1  -74717.742316   74764.470503  2023-09-26_3_Channel.C_012_Unit 1  
2 -108780.675539  108780.675524  2023-09-26_3_Channel.C_012_Unit 1  
3 -105763.678102  105763.678088  2023-09-26_3_Channel.C_012_Unit 1  
4  -74717.178424   74765.034389  2023-09-26_3_Channel.C_012_Unit 1  


## GLM Multiple Comparison Correction

In [5]:
# ---- multiple comparison correction ---
# select p-value column from glm_results
pvals = glm_results['P>|z|']
# correction method: 'fdr_bh' (False Discovery Rate, Benjamini/Hochberg)
reject, pvals_corrected, _, _ = multipletests(pvals, method='fdr_bh')

# add corrected p-value and reject to glm_results
glm_results['pval_corrected'] = pvals_corrected
glm_results['significant'] = reject

NameError: name 'glm_results' is not defined

In [23]:
# glm_results.to_excel('glm_results.xlsx')

In [24]:
plot_glm_coefficients(glm_results)

## Permutation ANOVA

In [52]:
zombies_trial_df= aggregate_trial_level(zombies_df)
# unique_neurons = zombies_trial_df['NeuronID'].unique()
# neuron_df = zombies_trial_df[zombies_trial_df['NeuronID'] == unique_neurons[0]]
zombies_trial_df.head()

,NeuronID,TaskField,MonkeyName,MonkeyGroup,SpikeCount
0,2023-09-26_3_Channel.C_002_Unit 1,1695753699241000,143H,Zombies,4
1,2023-09-26_3_Channel.C_002_Unit 1,1695753699414000,143H,Zombies,1
2,2023-09-26_3_Channel.C_002_Unit 1,1695753699562000,7124,Zombies,0
3,2023-09-26_3_Channel.C_002_Unit 1,1695753699874000,110E,Zombies,1
4,2023-09-26_3_Channel.C_002_Unit 1,1695753699967000,7124,Zombies,7


In [26]:
perm_anova_results = run_permutation_anova(zombies_trial_df, category_col='MonkeyName', plot=False)

Running permutation ANOVA per neuron: 100%|██████████| 32/32 [00:07<00:00,  4.38it/s]


Results:
                             NeuronID  F-statistic  p-value
0   2023-09-26_3_Channel.C_002_Unit 1     5.116338    0.000
1          2023-09-26_3_Channel.C_004     1.186998    0.336
2          2023-09-26_3_Channel.C_006     0.472202    0.913
3   2023-09-26_3_Channel.C_007_Unit 1     0.761809    0.620
4   2023-09-26_3_Channel.C_007_Unit 2     0.990568    0.433
5   2023-09-26_3_Channel.C_009_Unit 1     0.783351    0.608
6   2023-09-26_3_Channel.C_009_Unit 2     1.761453    0.114
7          2023-09-26_3_Channel.C_010     0.818452    0.902
8          2023-09-26_3_Channel.C_011     0.848581    0.542
9   2023-09-26_3_Channel.C_012_Unit 1     1.091445    0.340
10  2023-09-26_3_Channel.C_012_Unit 2     0.946060    0.470
11         2023-09-26_3_Channel.C_013     1.673057    0.114
12  2023-09-26_3_Channel.C_014_Unit 1     0.676258    0.771
13  2023-09-26_3_Channel.C_014_Unit 2     0.361645    0.947
14  2023-09-26_3_Channel.C_017_Unit 1     0.433197    0.912
15  2023-09-26_3_Channel.C_017

In [27]:
plot_permutation_anova_results_summary(perm_anova_results)

In [28]:
merge_glm_and_permutation_anova(glm_results, perm_anova_results)

,NeuronID,GLM_p-value,F-statistic,PermANOVA_p-value,GLM_pval_corrected,GLM_significant,Permutation_pval_corrected,Permutation_significant
0,2023-09-26_3_Channel.C_002_Unit 1,6.692064e-37,5.116338,0.000,1.505714e-36,True,0.000000,True
1,2023-09-26_3_Channel.C_004,6.859976e-32,1.186998,0.336,1.089526e-31,True,0.849150,False
2,2023-09-26_3_Channel.C_006,1.313429e-50,0.472202,0.913,1.182086e-49,True,0.947000,False
3,2023-09-26_3_Channel.C_007_Unit 1,4.380422e-49,0.761809,0.620,2.365428e-48,True,0.849150,False
4,2023-09-26_3_Channel.C_007_Unit 2,4.405745e-14,0.990568,0.433,5.947755e-14,True,0.849150,False
5,2023-09-26_3_Channel.C_009_Unit 1,4.223756e-108,0.783351,0.608,1.140414e-106,True,0.849150,False
6,2023-09-26_3_Channel.C_009_Unit 2,4.163036e-32,1.761453,0.114,7.480228e-32,True,0.513000,False
7,2023-09-26_3_Channel.C_010,9.993343e-01,0.818452,0.902,9.994115e-01,False,0.947000,False
8,2023-09-26_3_Channel.C_011,1.799178e-47,0.848581,0.542,6.072225e-47,True,0.849150,False
9,2023-09-26_3_Channel.C_012_Unit 1,9.994115e-01,1.091445,0.340,9.994115e-01,False,0.849150,False


In [33]:
glm_results.groupby('NeuronID')[['pval_corrected','P>|z|']].min().reset_index()

,NeuronID,pval_corrected,P>|z|
0,2023-09-26_3_Channel.C_002_Unit 1,1.355143e-35,6.692064e-37
1,2023-09-26_3_Channel.C_004,9.805730e-31,6.859976e-32
2,2023-09-26_3_Channel.C_006,1.063877e-48,1.313429e-50
3,2023-09-26_3_Channel.C_007_Unit 1,2.128885e-47,4.380422e-49
4,2023-09-26_3_Channel.C_007_Unit 2,4.866345e-13,4.405745e-14
5,2023-09-26_3_Channel.C_009_Unit 1,1.026373e-105,4.223756e-108
6,2023-09-26_3_Channel.C_009_Unit 2,6.732205e-31,4.163036e-32
7,2023-09-26_3_Channel.C_010,1.000000e+00,9.993343e-01
8,2023-09-26_3_Channel.C_011,5.465003e-46,1.799178e-47
9,2023-09-26_3_Channel.C_012_Unit 1,1.000000e+00,9.994115e-01
